# 05 Models — LightGBM — Men's

LightGBM as a fast alternative to XGBoost. Uses the same LOGO-CV framework with isotonic calibration.

**Inputs** (from S3 `04_preprocessing/mens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/lightgbm/mens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
except:
    !pip install lightgbm
    import lightgbm as lgb

from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

print(f"LightGBM version: {lgb.__version__}")

LightGBM version: 4.6.0


#### Functions

In [ ]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

def lgb_brier_eval(preds, train_data):
    """Custom Brier evaluation metric for LightGBM.
    With objective='binary', preds are already probabilities."""
    labels = train_data.get_label()
    brier = np.mean((preds - labels) ** 2)
    return 'brier', brier, False  # False = lower is better

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "05_models/lightgbm"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (5170, 103)
Stage 1 grid: (261013, 101)
Stage 2 grid: (66430, 100)
Features: 34


## 2. LightGBM Hyperparameters

In [ ]:
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'max_depth': 3,
    'num_leaves': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 10,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'seed': 42,
    'verbose': -1,
}

N_ROUNDS = 500
EARLY_STOPPING = 50

print("LightGBM parameters:")
for k, v in lgb_params.items():
    print(f"  {k}: {v}")

## 3. LOGO Cross-Validation

In [ ]:
train_original = train[train['TeamA'] < train['TeamB']].copy()
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols]
    y_train = train.loc[train_mask, 'Label']
    X_val = train_original.loc[val_mask, feature_cols]
    y_val = train_original.loc[val_mask, 'Label']
    
    if len(X_val) == 0:
        continue
    
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    callbacks = [
        lgb.early_stopping(EARLY_STOPPING, verbose=False),
        lgb.log_evaluation(period=0)
    ]
    
    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        valid_sets=[dval],
        feval=[lgb_brier_eval],
        callbacks=callbacks
    )
    
    # With objective='binary', predict returns probabilities directly
    preds = model.predict(X_val)
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': model.best_iteration
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}, BestRound={model.best_iteration}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

In [8]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")


Overall OOF Brier Score: 0.2372
Stage 1 (2022-2025) Brier Score: 0.2252
Mean Brier: 0.2371 +/- 0.0143


## 4. Train Final Model and Calibrate

In [ ]:
final_rounds = int(cv_df['BestRound'].median())
print(f"Final model rounds: {final_rounds}")

dtrain_all = lgb.Dataset(train[feature_cols], label=train['Label'])
final_model = lgb.train(lgb_params, dtrain_all, num_boost_round=final_rounds)

# Calibration
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

## 5. Generate Predictions

In [ ]:
# With objective='binary', predict returns probabilities directly
stage1['Pred_lightgbm'] = final_model.predict(stage1[feature_cols])
stage1['Pred_lightgbm_calibrated'] = calibrator.predict(stage1['Pred_lightgbm'].values).clip(0.02, 0.98)

stage2['Pred_lightgbm'] = final_model.predict(stage2[feature_cols])
stage2['Pred_lightgbm_calibrated'] = calibrator.predict(stage2['Pred_lightgbm'].values).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_lightgbm_calibrated'].min():.3f}, {stage1['Pred_lightgbm_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_lightgbm_calibrated'].min():.3f}, {stage2['Pred_lightgbm_calibrated'].max():.3f}]")

# Evaluate on actual Stage 1 games
stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_lightgbm_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

## 6. Feature Importance

In [12]:
imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Gain': final_model.feature_importance(importance_type='gain')
}).sort_values('Gain', ascending=False)

print("Feature Importance (gain):")
for _, row in imp_df.iterrows():
    print(f"  {row['Feature']:30s}: {row['Gain']:.1f}")

Feature Importance (gain):
  SeedDiff                      : 1877.2
  EloDiff                       : 914.1
  TopSystemsAvgRankDiff         : 354.2
  IsPowerConfDiff               : 152.0
  SeedB                         : 123.8
  SeedA                         : 110.3
  AvgPointDiffDiff              : 71.4
  MORDiff                       : 50.1
  RecentWinPctDiff              : 33.2
  AvgStlDiff                    : 31.8
  AvgTODiff                     : 30.2
  WinPctDiff                    : 26.9
  OffEffDiff                    : 18.0
  FTPctDiff                     : 16.5
  FGPctDiff                     : 14.2
  SAGDiff                       : 14.0
  WLKDiff                       : 12.3
  AvgPossDiff                   : 11.4
  AvgBlkDiff                    : 11.3
  POMDiff                       : 8.2
  OppFG3PctDiff                 : 8.2
  NetEffDiff                    : 6.7
  DefEffDiff                    : 6.4
  AvgORDiff                     : 5.9
  AvgAstDiff                    : 5

## 7. Save Outputs

In [13]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_lightgbm', 'Pred_lightgbm_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_lightgbm', 'Pred_lightgbm_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/lightgbm/mens/oof_predictions.parquet ((2585, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/lightgbm/mens/stage1_predictions.parquet ((261013, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/lightgbm/mens/stage2_predictions.parquet ((66430, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/lightgbm/mens/cv_results.parquet ((40, 4))


## 8. Summary

In [14]:
print("=" * 60)
print("LIGHTGBM MODEL SUMMARY — MEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
print(f"CV Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"Final model rounds: {final_rounds}")

LIGHTGBM MODEL SUMMARY — MEN'S

OOF Brier Score (raw): 0.2372
OOF Brier Score (calibrated): 0.1805
CV Mean Brier: 0.2371 +/- 0.0143
Final model rounds: 73
